# ArcFace

In [7]:
!pip3 install insightface onnxruntime opencv-python
import cv2, torch, insightface, os
import numpy as np
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image
from torch.nn.functional import cosine_similarity
print("Current working directory:", os.getcwd())

# Load ResNet100
app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=-1, det_size=(640, 640))

Current working directory: g:\.thesis\named-ai\data-preprocessing
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\mode

In [ ]:
#get embeddings FIX ERROR HANDLING
def get_embeddings_resnet100(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found or unreadable: {image_path}")

    faces = app.get(img)
    if len(faces) == 0:
        print(f"No face detected in {image_path}")
        return None
    print(f"Face detected in {image_path}")
    emb = torch.tensor(faces[0].embedding).squeeze()
    emb = torch.nn.functional.normalize(emb, p=2, dim=0)
    return emb

In [17]:
def average_embeddings(image_paths):
    embeddings = []
    for path in image_paths:
        emb = get_embeddings_resnet100(path)
        if emb is not None:
            embeddings.append(emb)

    if not embeddings:
        return None  # no valid embeddings

    avg_embedding = torch.stack(embeddings).mean(dim=0)
    # Normalize again after averaging
    avg_embedding = torch.nn.functional.normalize(avg_embedding, p=2, dim=0)
    return avg_embedding

In [18]:
#recognize face FIX ERROR HANDLING
def recognize_face(test_embedding, face_db, threshold=0.6):
    if test_embedding is None:
        return "Unknown", 0.0

    max_sim = 0
    identity = "Unknown"

    for name, db_embedding in face_db.items():
        sim = cosine_similarity(test_embedding.unsqueeze(0), db_embedding.unsqueeze(0))
        sim_val = sim.item()

        if sim_val > max_sim and sim_val > threshold:
            max_sim = sim_val
            identity = name

    return identity, max_sim

In [43]:
print("Current working directory:", os.getcwd())
face_db_r100 = {}
#db_images = {}
#test_images = {}
    #"Akshay": ["orig-img/orig-img/akshay/Akshay Kumar_3.jpg", "orig-img/orig-img/akshay/Akshay Kumar_1.jpg", "orig-img/orig-img/akshay/Akshay Kumar_2.jpg"],
    #"Alexandra": ["orig-img/orig-img/alexandra/Alexandra Daddario_0.jpg", "orig-img/orig-img/alexandra/Alexandra Daddario_1.jpg", "orig-img/orig-img/alexandra/Alexandra Daddario_2.jpg"]
base_dir = "final-images"
face_db = {}
people = {
    "Alexandra Daddario": "Alexandra Daddario",
    "Andy Samberg": "Andy Samberg",
    "Brad Pitt": "Brad Pitt",
    "Camila Cabello": "Camila Cabello",
    "The Rock": "Dwayne Johnson",
    "Elizabeth Olsen": "Elizabeth Olsen",
    "Henry Cavill": "Henry Cavill",
    "Margot Robbie": "Margot Robbie",
    "Robert Downey Jr.": "Robert Downey Jr",   
    "Tom Cruise": "Tom Cruise"
}

for name, folder in people.items():
    folder_path = os.path.join(base_dir, folder)
    image_files = [
        os.path.join(folder_path, f"{folder}_{i}.jpg")
        for i in range(45)   # 0–45 inclusive
    ]
    #image_files = [os.path.join(folder_path, f)
    #               for f in os.listdir(folder_path)
     #              if f.lower().endswith((".jpg", ".jpeg", ".png"))
    #]

    db_embeddings = []
    for i in range(min(45,len(image_files))):
        emb = get_embeddings_resnet100(image_files[i])
        if emb is not None:
            db_embeddings.append(emb)
    if db_embeddings:
        avg_emb = torch.stack(db_embeddings).mean(dim=0)
        avg_emb = torch.nn.functional.normalize(avg_emb, p=2, dim=0)
        face_db[name] = avg_emb

    #db_images[name] = image_files[:45]
    #test_images[name] = image_files[45:]

#for name, paths in db_images.items():
#    avg_emb = average_embeddings(paths)
#    if avg_emb is not None:
#        face_db[name] = avg_emb
#for name, paths in known_faces.items():
#    if isinstance(paths, str):
#        paths = [paths]  # handle single string input
#    avg_emb = average_embeddings(paths)
#    if avg_emb is not None:
#        face_db_r100[name] = avg_emb
# build db
#face_db_r100 = {}
#face_db_r100["Kyle"] = get_embeddings_resnet100("my_images/kyle_183.jpg")
#face_db_r100["Tom Cruise"] = get_embeddings_resnet100("my_images/Tom Cruise_12.jpg")
#face_db_r100["Tom CruiseTest"] = get_embeddings_resnet100("my_images/testtom.jpg")
#print("Current working directory:", os.getcwd())
#face_db_r100["Kumar"] = get_embeddings_resnet100("orig-img/orig-img/akshay/Akshay Kumar_0.jpg")

Current working directory: g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_0.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_1.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_2.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_3.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_4.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_5.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_6.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_7.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_8.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_9.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_10.jpg
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_11.jpg
Face detected in 

In [42]:
for name, paths in test_images.items():
    print(f"\nTesting recognition for {name}:")
    for path in paths:
        test_emb = get_embeddings_resnet100(path)
        identity, confidence = recognize_face(test_emb, face_db)
        print(f"{os.path.basename(path)} recognized as {identity} (conf ={confidence:.2f})")


Testing recognition for Alexandra Daddario:
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_5.jpg
Alexandra Daddario_5.jpg recognized as Alexandra Daddario (conf =0.87)
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_50.jpg
Alexandra Daddario_50.jpg recognized as Alexandra Daddario (conf =0.75)
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_51.jpg
Alexandra Daddario_51.jpg recognized as Alexandra Daddario (conf =0.74)
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_52.jpg
Alexandra Daddario_52.jpg recognized as Alexandra Daddario (conf =0.86)
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_53.jpg
Alexandra Daddario_53.jpg recognized as Alexandra Daddario (conf =0.89)
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_54.jpg
Alexandra Daddario_54.jpg recognized as Alexandra Daddario (conf =0.89)
Face detected in final-images\Alexandra Daddario\Alexandra Daddario_55.jpg


In [40]:
#test recognition
#test_embedding_r100 = get_embeddings_resnet100("my_images/testtom.jpg")
test_embedding_r100 = get_embeddings_resnet100("final-images/Elizabeth Olsen/Elizabeth Olsen_5.jpg")

print(f"{test_embedding_r100}")
identity_r100, confidence_r100 = recognize_face(test_embedding_r100, face_db, threshold=0.6)
print(f"[ResNet100] Identified as: {identity_r100} (Confidence: {confidence_r100:.2f})")


Face detected in final-images/Elizabeth Olsen/Elizabeth Olsen_5.jpg
tensor([ 1.1082, -1.3418,  0.7403, -0.7802, -0.5886,  0.5704, -1.2897, -0.4062,
         1.4853, -0.7576, -0.6075,  1.3075,  0.3553,  0.3129,  0.5582,  1.9845,
         0.5733, -0.2624,  1.8874,  1.1633,  0.2261,  2.5953, -1.2080, -0.3324,
        -0.4138, -1.0801,  0.1993, -0.0653, -0.0522,  0.1154,  0.6185,  0.2700,
         0.6266,  0.8826, -0.2305,  0.0444,  1.1443, -2.0220,  1.1854, -1.1019,
        -1.0305,  1.9668, -0.6220,  0.0788, -0.5913, -0.2131, -0.3177, -0.1857,
         0.7752, -0.9840,  0.1828,  0.8254, -0.1036, -1.3297,  0.7042,  0.3800,
        -0.0102, -0.0944, -2.2843, -0.3485, -1.9751, -2.3123, -0.1490, -0.4574,
         0.5924, -0.3541,  0.2060, -0.6736, -0.9284, -1.1254, -0.9933,  0.0382,
         0.1848, -1.4747, -0.9984,  0.3995,  2.4146, -1.0779, -0.1192, -0.9081,
        -0.3611, -0.6023,  1.2129, -0.3587,  0.5197,  0.4900, -0.1081,  1.4232,
         0.2142, -1.8294, -2.6038,  0.9003, -0.9542,